In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
DATA_FILE = "raw_skills.csv"
TOP_N = 3
MIN_SKILLS = 3

In [5]:

def load_dataset(path: str) -> pd.DataFrame:
    
    df = pd.read_csv(path)

    is_missing = df["skills"].isna() | (df["skills"].str.strip() == "")
    df["is_cold_start_item"] = is_missing

    df.loc[is_missing, "skills"] = df.loc[is_missing, "role"].str.replace(
        r"[^A-Za-z0-9 ]", "", regex=True
    )

    df["skills"] = df["skills"].str.strip()
    return df

In [6]:

def get_user_input(min_skills: int = MIN_SKILLS) -> list[tuple[str, int]]:
    
    print(f"\nEnter at least {min_skills} skills, ordered from MOST to LEAST important.")
    print("Example: Python, Cloud Computing, Automation\n")

    while True:
        raw = input("Your skills (priority order): ").strip()
        skills = [s.strip() for s in raw.split(",") if s.strip()]
        if len(skills) >= min_skills:
            break
        print(f"Please enter at least {min_skills} skills. You entered {len(skills)}.")

    n = len(skills)
    weighted_skills = [(skill, n - idx) for idx, skill in enumerate(skills)]

    print("\nPriority weights assigned:")
    for skill, weight in weighted_skills:
        print(f"  {skill}: weight {weight}")

    return weighted_skills


In [7]:

def build_weighted_profile(weighted_skills: list[tuple[str, int]]) -> str:
    
    tokens = []
    for skill, weight in weighted_skills:
        tokens.extend([skill] * weight)
    return ", ".join(tokens)


In [8]:

def build_similarity_scores(df: pd.DataFrame, weighted_skills: list[tuple[str, int]]) -> pd.DataFrame:
    
    user_profile = build_weighted_profile(weighted_skills)

    corpus = df["skills"].tolist() + [user_profile]
    vectorizer = TfidfVectorizer(token_pattern=r"[A-Za-z0-9\+\.#/]+")
    tfidf_matrix = vectorizer.fit_transform(corpus)

    role_vectors = tfidf_matrix[:-1]   
    user_vector = tfidf_matrix[-1]     

    scores = cosine_similarity(user_vector, role_vectors).flatten()

    result = df.copy()
    result["similarity_score"] = scores
    return result


In [9]:

def rank_and_filter(scored_df: pd.DataFrame, top_n: int = TOP_N) -> pd.DataFrame:
    
    ranked = scored_df.sort_values("similarity_score", ascending=False)
    return ranked.head(top_n)

In [10]:

def cold_start_fallback(df: pd.DataFrame, top_n: int = TOP_N) -> pd.DataFrame:
    
    print("\n[Notice] No meaningful skill overlap found - showing trending roles instead.\n")
    fallback = df.copy()
    fallback["similarity_score"] = 0.0
    return fallback.head(top_n)

In [11]:

def display_results(results: pd.DataFrame) -> None:
    
    print("\n" + "=" * 50)
    print(" TOP RECOMMENDED CAREER PATHS")
    print("=" * 50)
    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        print(f"{rank}. {row['role']}  (match score: {row['similarity_score']:.2f})")
        print(f"   Key skills: {row['skills']}")
        if row.get("is_cold_start_item", False):
            print("   Note: new role with limited skill data (item cold start - "
                  "ranked using role name only, low confidence)")
    print("=" * 50 + "\n")


In [12]:
def main():
    df = load_dataset(DATA_FILE)
    weighted_skills = get_user_input()

    scored_df = build_similarity_scores(df, weighted_skills)

    if scored_df["similarity_score"].max() == 0:
        top_results = cold_start_fallback(df)
    else:
        top_results = rank_and_filter(scored_df)

    display_results(top_results)

In [14]:
if __name__ == "__main__":
    main()


Enter at least 3 skills, ordered from MOST to LEAST important.
Example: Python, Cloud Computing, Automation



Your skills (priority order):  java, data analyst, automation



Priority weights assigned:
  java: weight 3
  data analyst: weight 2
  automation: weight 1

 TOP RECOMMENDED CAREER PATHS
1. Backend Developer  (match score: 0.26)
   Key skills: Java, Python, SQL, APIs, Databases, System Design
2. Mobile App Developer  (match score: 0.21)
   Key skills: Java, Kotlin, Swift, UI Design, APIs, Mobile Development
3. Data Analyst  (match score: 0.19)
   Key skills: SQL, Excel, Data Analysis, Python, Statistics, Data Visualization

